# 06 High Frequency Data

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AmirrezaFarnamTaheri/Computational-Economics-and-Data-Science/blob/main/09-Finance/06_High_Frequency_Data.ipynb) [![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/AmirrezaFarnamTaheri/Computational-Economics-and-Data-Science/main?filepath=09-Finance/06_High_Frequency_Data.ipynb) [![Code License: MIT](https://img.shields.io/badge/Code%20License-MIT-yellow.svg)](../LICENSE) [![Content License: CC BY 4.0](https://img.shields.io/badge/Content%20License-CC%20BY%204.0-blue.svg)](https://creativecommons.org/licenses/by/4.0/)


In [ ]:
rng = np.random.default_rng(42)  # single reproducible generator

# === Environment Setup ===
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# --- Configuration ---
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 130, 'font.size': 12, 'axes.titlesize': 'x-large',
    'axes.labelsize': 'large', 'xtick.labelsize': 'medium', 'ytick.labelsize': 'medium'})
%config InlineBackend.figure_format = 'retina'


### Table of Contents

1.  [Introduction: The World of Ticks and Trades](#1.-Introduction:-The-World-of-Ticks-and-Trades)
2.  [Handling Tick Data](#2.-Handling-Tick-Data)
3.  [The Limit Order Book (LOB) and Key Concepts](#3.-The-Limit-Order-Book-(LOB)-and-Key-Concepts)
4.  [Realized Volatility and Microstructure Noise](#4.-Realized-Volatility-and-Microstructure-Noise)
5.  [Order Flow Imbalance (OFI)](#5.-Order-Flow-Imbalance-(OFI))
6.  [Summary](#6.-Summary)


## The Lens: Finance at the Speed of Light
**What economic problem are we solving?**
Standard finance models operate in daily or monthly time. But markets operate in microseconds. In the blink of an eye, prices crash and recover, liquidity evaporates, and algorithms trade millions of shares. We need to solve the problem of **microstructure**: how the mechanics of trading itself affects prices, liquidity, and volatility.

**Why do we need this method?**
High-frequency data is noisy, massive, and messy. Standard econometric tools fail here. We need specialized techniques to handle the **Limit Order Book (LOB)**, filter out microstructure noise to measure **Realized Volatility**, and understand the **Order Flow** that ultimately moves prices.



**Economic question.** In *06 High Frequency Data*, what must remain economically invariant when the computational representation changes? Financial models connect prices to risk, timing, and no-arbitrage restrictions. The economic question is which states of the world make a payoff valuable, how risk is transferred or hedged, and how sensitive the valuation is to assumptions about dynamics and market completeness. A numerical price is credible only when it respects basic bounds, limiting cases, and an independent replication or hedging argument.

### Learning Objectives
* **Process** tick-level data: timestamps, irregular spacing, and trade/quote alignment.
* **Construct** and visualize the Limit Order Book and measure bid-ask spreads.
* **Estimate** realized volatility using sub-sampling and kernel-based estimators.
* **Analyze** order flow imbalance and its predictive power for short-horizon returns.

### Prerequisites
* **Volatility Modeling:** GARCH and conditional variance (Module 08 - ARCH/GARCH).
* **Statistics:** Variance estimation, sampling theory, and noise filtering.
* **Python:** Pandas time-series indexing, resampling, and handling large DataFrames.
* **Learning-path prerequisite:** [`05_Credit_Risk.ipynb`](05_Credit_Risk.ipynb)


> **Learning path:** Building on [`05_Credit_Risk.ipynb`](05_Credit_Risk.ipynb); next continue with [`07_Financial_Frictions_BGG.ipynb`](07_Financial_Frictions_BGG.ipynb).


### 1. Introduction: The World of Ticks and Trades

This notebook introduces the world of **high-frequency data** and **market microstructure**—the study of how transaction costs, information asymmetry, and the detailed mechanics of trading affect asset prices. We move from the daily or monthly data used in traditional asset pricing to the level of individual trades and quotes, which occur in milliseconds or even microseconds.


### 2. Handling Tick Data
High-frequency data, often called **tick data**, records every single event in the market. This requires specialized data handling techniques due to its large volume and irregular time intervals.


In [ ]:
### Creating Synthetic Tick Data
n_ticks = 10000
base_time = pd.to_datetime('2023-10-27 09:30:00')
time_deltas = np.random.exponential(scale=0.1, size=n_ticks).cumsum()
timestamps = base_time + pd.to_timedelta(time_deltas, unit='s')

mid_price = 100 + rng.standard_normal(n_ticks).cumsum() * 0.01
spread = rng.uniform(0.01, 0.05, n_ticks)
bid = mid_price - spread / 2
ask = mid_price + spread / 2
volume = rng.integers(100, 1000, n_ticks)

tick_df = pd.DataFrame({'bid': bid, 'ask': ask, 'volume': volume}, index=timestamps)
print("> **Note:** Synthetic tick data created.")
display(tick_df.head())


### 3. The Limit Order Book (LOB) and Key Concepts

Modern electronic markets are organized around a **Limit Order Book (LOB)**. This is a centralized ledger of all outstanding buy (bid) and sell (ask) orders for a given asset, organized by price level. The diagram below shows a snapshot of a typical LOB.

![Limit Order Book](../images/09-Finance/limit_order_book.png)

From the LOB, we can derive several key microstructure concepts:

- **Best Bid and Best Ask:** The highest bid price and the lowest ask price available in the market at any given moment.
- **Bid-Ask Spread:** The difference between the best ask and the best bid ($Spread = Ask_{best} - Bid_{best}$). It is a primary measure of market liquidity and a key cost for traders who demand immediate execution by crossing the spread.
- **Market Depth:** The volume of orders available at the best bid and ask, and at other price levels. Deeper markets are more liquid as they can absorb larger trades without a significant price impact.
- **Volume-Weighted Average Price (VWAP):** The average price of a stock over a given time period, weighted by the volume of trades at each price. It is often used as a benchmark for execution quality. A trader who buys below the VWAP has achieved a good execution.


In [ ]:
### Calculating Microstructure Metrics
tick_df['spread'] = tick_df['ask'] - tick_df['bid']
tick_df['midprice'] = (tick_df['ask'] + tick_df['bid']) / 2

# Calculate VWAP over 5-minute intervals
tick_df['vwap'] = (tick_df['midprice'] * tick_df['volume']).resample('5T').sum() / tick_df['volume'].resample('5T').sum()
tick_df['vwap'].fillna(method='ffill', inplace=True)

print("> **Note:** Calculated spread and VWAP.")
display(tick_df[['midprice', 'vwap', 'spread']].head())

### Visualizing Midprice vs. VWAP
plt.figure(figsize=(14, 7))
plt.plot(tick_df.index, tick_df['midprice'], label='Mid-Price (Tick-by-Tick)', alpha=0.6) # Plot data series
plt.plot(tick_df.index, tick_df['vwap'], label='VWAP (5-Minute)', color='red', linestyle='--', lw=2.5) # Plot data series
plt.title('High-Frequency Mid-Price vs. 5-Minute VWAP')
plt.xlabel('Time')
plt.ylabel('Price')
plt.legend()
plt.show() # Render plot


### 4. Realized Volatility and Microstructure Noise

One of the most powerful applications of high-frequency data is the ability to compute a precise, model-free measure of volatility. While GARCH models *estimate* conditional volatility based on daily data, **realized volatility** *measures* the actual volatility that occurred over a period (e.g., a day) by summing the squared high-frequency returns.

If we have intraday returns $r_{t,j}$ (e.g., from 5-minute intervals), the realized variance for day $t$ is:
$$ RV_t = \sum_{j=1}^{M} r_{t,j}^2 $$
The annualized realized volatility is then $\sqrt{252 \cdot RV_t}$. This provides a much more accurate measure of daily volatility than can be obtained from daily data alone.

**The Challenge of Microstructure Noise:**
In theory, the more frequently we sample returns (e.g., every second), the more accurate our realized volatility measure should be. In practice, this is not true. At very high frequencies, the observed prices are contaminated by **market microstructure noise**, such as the price bouncing between the bid and ask prices. This noise adds variance that is not part of the true price process.

This leads to a trade-off: sampling too infrequently means we miss some of the true price variation, while sampling too frequently means our measure is contaminated by noise. The **volatility signature plot** is the standard diagnostic tool for visualizing this trade-off and choosing an appropriate sampling frequency. It plots the average realized volatility against the sampling interval.


In [ ]:
### Calculating Realized Volatility and Signature Plot

def calculate_realized_vol(data, freq):
    """Calculates realized volatility for a given sampling frequency."""
    returns = data['midprice'].resample(freq).last().pct_change().dropna()
    rv_daily = (returns**2).resample('D').sum()
    # Return average annualized vol
    return np.sqrt(np.mean(rv_daily) * 252)

# Calculate RV for a range of frequencies
frequencies = [f'{s}S' for s in range(10, 61, 5)] + [f'{m}T' for m in range(2, 31, 2)]
volatilities = [calculate_realized_vol(tick_df, freq) for freq in frequencies]

print("> **Note:** Calculating realized volatility across different sampling frequencies...")
plt.figure(figsize=(14, 7))
plt.plot([pd.to_timedelta(f).total_seconds() for f in frequencies], volatilities, marker='o') # Plot data series
plt.title('Volatility Signature Plot')
plt.xlabel('Sampling Interval (seconds)')
plt.ylabel('Average Annualized Realized Volatility')
plt.xscale('log')
plt.show() # Render plot

print("> **Note:** The signature plot shows the classic U-shape. At very high frequencies (left), the measured volatility is high due to microstructure noise. As we sample less frequently, the noise effect diminishes and the volatility estimate drops. At very low frequencies (right), we start to miss true price variation, so the estimate may begin to drift. A common choice for the optimal frequency is the minimum of this curve, often around 5 minutes for liquid stocks.")


### 5. Order Flow Imbalance (OFI)

A more modern microstructure concept is **Order Flow Imbalance (OFI)**. It measures the net pressure on the bid and ask sides of the book, capturing the intensity of buying versus selling interest. A simplified version can be calculated based on changes in the best bid and ask prices and sizes.

$$ OFI_t = I_{B,t} - I_{A,t} $$ 
Where $I_{B,t}$ is an indicator for buying pressure and $I_{A,t}$ is for selling pressure. For example:
- $I_{B,t} = \Delta q_{B,t}$ if $\Delta p_{B,t} \ge 0$ (volume increases at a non-decreasing price)
- $I_{A,t} = \Delta q_{A,t}$ if $\Delta p_{A,t} \le 0$ (volume increases at a non-increasing price)

A positive OFI indicates strong buying pressure and has been shown to predict short-term price increases.


In [ ]:
### Calculating Order Flow Imbalance
df = tick_df.copy()
df['prev_bid_price'] = df['bid'].shift(1)
df['prev_ask_price'] = df['ask'].shift(1)
df['prev_bid_size'] = df['volume'].shift(1) # Using total volume as a proxy
df['prev_ask_size'] = df['volume'].shift(1) # Using total volume as a proxy

df['delta_bid_price'] = df['bid'].diff()
df['delta_ask_price'] = df['ask'].diff()
df['delta_bid_size'] = df['volume'].diff()
df['delta_ask_size'] = df['volume'].diff()

I_B = np.where(df['delta_bid_price'] >= 0, df['delta_bid_size'], 0)
I_A = np.where(df['delta_ask_price'] <= 0, df['delta_ask_size'], 0)

df['OFI'] = I_B - I_A

# Plot cumulative OFI against price
df['cumulative_OFI'] = df['OFI'].cumsum().fillna(0)

fig, ax1 = plt.subplots(figsize=(14, 8))
ax1.plot(df.index, df['midprice'], 'b-', label='Mid-Price')
ax1.set_xlabel('Time')
ax1.set_ylabel('Price', color='b')
ax1.tick_params('y', colors='b')

ax2 = ax1.twinx()
ax2.plot(df.index, df['cumulative_OFI'], 'r-', alpha=0.6, label='Cumulative OFI')
ax2.set_ylabel('Cumulative Order Flow Imbalance', color='r')
ax2.tick_params('y', colors='r')

fig.legend(loc='upper left', bbox_to_anchor=(0.1, 0.9))
plt.title('Mid-Price vs. Cumulative Order Flow Imbalance')
plt.show() # Render plot
print("> **Note:** The plot shows a strong correlation between the cumulative OFI and the mid-price, illustrating how net buying pressure drives prices up.")


## Exercises

**1. Mechanism and assumptions (Conceptual):** Derive the no-arbitrage, optimality, or risk-pricing relation central to **06 High Frequency Data** and verify that it satisfies at least two economically meaningful limiting cases or bounds.

**2. Reproduce and diagnose (Applied):** Reproduce a calculation from 1. Introduction: The World of Ticks and Trades, 2. Handling Tick Data with transparent inputs. Perturb volatility, discounting, risk aversion, transaction costs, or another key parameter and explain the sensitivity in economic terms.

**3. Robust extension (Challenge):** Construct a stress scenario outside the calibration sample. Compare two valuation/risk methods and explain which discrepancy reflects model risk rather than numerical error.

<details>
<summary>Solution guidance</summary>

A strong solution states assumptions before computation, includes an independent diagnostic or limiting-case check, and interprets the result in the units of the economic problem. For the challenge, separate changes caused by the economic assumption from changes caused by numerical approximation or tuning.

</details>


## Key Equations

- **Log return:** $r_{t,i}=\log P_{t,i}-\log P_{t,i-1}$.
- **Realized variance:** $RV_t=\sum_i r_{t,i}^2$ over intraday intervals.
- **Bid-ask spread:** quoted spread is ask minus bid; effective spread uses the trade price relative to the prevailing midpoint.
- **Microstructure caution:** as sampling becomes too fine, bid-ask bounce and price discreteness contaminate naive realized-variance estimates.


---
## Summary

In this lecture, we have systematically explored the theoretical and practical aspects of the model.

**Key Takeaways:**
1.  **Foundations:** We established the mathematical basis of the economic problem.
2.  **Computation:** We implemented the solution using efficient algorithms.
3.  **Implications:** We analyzed the economic significance of the results.

**Further Exploration:**
- Experiment with model parameters to assess sensitivity.
- Extend the framework by relaxing simplifying assumptions.


## References & Further Reading

- Cochrane, J. H. (2005). *Asset Pricing* (rev. ed.). Princeton University Press.
- Campbell, J. Y., Lo, A. W. & MacKinlay, A. C. (1997). *The Econometrics of Financial Markets*. Princeton University Press.
- Shreve, S. E. (2004). *Stochastic Calculus for Finance II*. Springer.
